# Image Fundamentals

This notebook covers the basics of digital image processing:
1. Images as NumPy arrays
2. RGB channels and pixel histograms
3. Basic operations: resize, crop, rotate, flip
4. Color space conversions
5. Edge detection with Sobel filters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image, ImageDraw, ImageFilter
from scipy import ndimage

plt.rcParams['figure.figsize'] = (10, 6)
print('Imports ready.')

## 1. Creating Synthetic Test Images

We generate test images with NumPy so no external files are needed.

In [ ]:
def create_test_image(size=256):
    """Create a colorful synthetic test image with shapes."""
    img = np.zeros((size, size, 3), dtype=np.uint8)
    
    # Blue gradient background
    for i in range(size):
        img[i, :, 2] = int(50 + 150 * i / size)
    
    # Red circle
    y, x = np.ogrid[:size, :size]
    cx, cy, r = size//4, size//4, size//6
    mask = (x - cx)**2 + (y - cy)**2 < r**2
    img[mask] = [220, 50, 50]
    
    # Green rectangle
    img[size//2:size//2+size//4, size//2:size//2+size//3] = [50, 200, 50]
    
    # Yellow triangle
    for i in range(size//5):
        left = size//6 + i
        right = size//6 + size//5 - i
        row = size//2 + size//4 + i
        if row < size and left < size and right < size:
            img[row, left:right] = [240, 220, 50]
    
    # White diagonal line
    for i in range(size):
        if i < size:
            img[i, min(i, size-1)] = [255, 255, 255]
            if i+1 < size:
                img[i, min(i+1, size-1)] = [255, 255, 255]
    
    return img

test_img = create_test_image(256)
print(f'Image shape: {test_img.shape}')
print(f'Data type: {test_img.dtype}')
print(f'Value range: [{test_img.min()}, {test_img.max()}]')

plt.figure(figsize=(6, 6))
plt.imshow(test_img)
plt.title(f'Synthetic Test Image ({test_img.shape[0]}x{test_img.shape[1]}x{test_img.shape[2]})')
plt.axis('off')
plt.show()

## 2. RGB Channels

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].imshow(test_img)
axes[0].set_title('Original (RGB)')
axes[0].axis('off')

channel_names = ['Red', 'Green', 'Blue']
cmaps = ['Reds', 'Greens', 'Blues']

for i in range(3):
    axes[i+1].imshow(test_img[:, :, i], cmap=cmaps[i])
    axes[i+1].set_title(f'{channel_names[i]} Channel')
    axes[i+1].axis('off')

plt.suptitle('RGB Channel Decomposition', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Pixel Value Histograms

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-channel histogram
colors = ['red', 'green', 'blue']
for i, color in enumerate(colors):
    axes[0].hist(test_img[:, :, i].ravel(), bins=64, color=color, alpha=0.5, label=color.capitalize())
axes[0].set_title('Per-Channel Pixel Histogram')
axes[0].set_xlabel('Pixel Value')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Grayscale histogram
gray = np.mean(test_img, axis=2)
axes[1].hist(gray.ravel(), bins=64, color='gray', edgecolor='black', linewidth=0.5)
axes[1].set_title('Grayscale Pixel Histogram')
axes[1].set_xlabel('Pixel Value')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 4. Basic Image Operations

### Resize, Crop, Rotate, Flip

In [ ]:
pil_img = Image.fromarray(test_img)

operations = {
    'Original': pil_img,
    'Resized (128x128)': pil_img.resize((128, 128)),
    'Cropped (center)': pil_img.crop((64, 64, 192, 192)),
    'Rotated 45deg': pil_img.rotate(45, expand=False, fillcolor=(0, 0, 0)),
    'Flipped Horizontal': pil_img.transpose(Image.FLIP_LEFT_RIGHT),
    'Flipped Vertical': pil_img.transpose(Image.FLIP_TOP_BOTTOM),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, (name, img) in zip(axes.flat, operations.items()):
    ax.imshow(np.array(img))
    ax.set_title(name)
    ax.axis('off')

plt.suptitle('Basic Image Operations', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Color Space Conversions

In [ ]:
# RGB to Grayscale (weighted average)
gray_weighted = 0.299 * test_img[:,:,0] + 0.587 * test_img[:,:,1] + 0.114 * test_img[:,:,2]

# RGB to HSV using matplotlib
img_float = test_img.astype(np.float64) / 255.0
hsv_img = mcolors.rgb_to_hsv(img_float)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].imshow(test_img)
axes[0, 0].set_title('Original RGB')
axes[0, 0].axis('off')

axes[0, 1].imshow(gray_weighted, cmap='gray')
axes[0, 1].set_title('Grayscale\n(0.299R + 0.587G + 0.114B)')
axes[0, 1].axis('off')

# Binary thresholding
threshold = 128
binary = (gray_weighted > threshold).astype(np.uint8) * 255
axes[0, 2].imshow(binary, cmap='gray')
axes[0, 2].set_title(f'Binary (threshold={threshold})')
axes[0, 2].axis('off')

# HSV channels
hsv_names = ['Hue', 'Saturation', 'Value']
hsv_cmaps = ['hsv', 'gray', 'gray']
for i in range(3):
    axes[1, i].imshow(hsv_img[:, :, i], cmap=hsv_cmaps[i])
    axes[1, i].set_title(f'HSV: {hsv_names[i]}')
    axes[1, i].axis('off')

plt.suptitle('Color Space Conversions', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Edge Detection with Sobel Filters

The Sobel operator computes the gradient of image intensity at each pixel, highlighting edges.

In [ ]:
# Define Sobel kernels
sobel_x = np.array([[-1, 0, 1],
                     [-2, 0, 2],
                     [-1, 0, 1]], dtype=np.float64)

sobel_y = np.array([[-1, -2, -1],
                     [ 0,  0,  0],
                     [ 1,  2,  1]], dtype=np.float64)

print('Sobel X kernel (horizontal edges):')
print(sobel_x)
print('\nSobel Y kernel (vertical edges):')
print(sobel_y)

In [ ]:
# Apply Sobel filters to grayscale image
edges_x = ndimage.convolve(gray_weighted, sobel_x)
edges_y = ndimage.convolve(gray_weighted, sobel_y)
edges_magnitude = np.sqrt(edges_x**2 + edges_y**2)

fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0, 0].imshow(gray_weighted, cmap='gray')
axes[0, 0].set_title('Original (Grayscale)')
axes[0, 0].axis('off')

axes[0, 1].imshow(np.abs(edges_x), cmap='gray')
axes[0, 1].set_title('Sobel X (Vertical Edges)')
axes[0, 1].axis('off')

axes[1, 0].imshow(np.abs(edges_y), cmap='gray')
axes[1, 0].set_title('Sobel Y (Horizontal Edges)')
axes[1, 0].axis('off')

axes[1, 1].imshow(edges_magnitude, cmap='hot')
axes[1, 1].set_title('Edge Magnitude')
axes[1, 1].axis('off')

plt.suptitle('Sobel Edge Detection', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Create a more detailed test image for better edge demo
def create_edge_test_image(size=256):
    """Create an image with clear edges for edge detection demos."""
    img = np.ones((size, size), dtype=np.float64) * 200
    
    # Checkerboard pattern region
    block = 32
    for i in range(0, size//2, block):
        for j in range(0, size//2, block):
            if (i//block + j//block) % 2 == 0:
                img[i:i+block, j:j+block] = 50
    
    # Concentric circles
    y, x = np.ogrid[:size, :size]
    cx, cy = 3*size//4, 3*size//4
    for r in range(20, size//3, 20):
        ring = ((x-cx)**2 + (y-cy)**2 > (r-5)**2) & ((x-cx)**2 + (y-cy)**2 < r**2)
        img[ring] = 30
    
    # Gradient region
    img[size//2:, :size//2] = np.tile(np.linspace(0, 255, size//2), (size//2, 1))
    
    return img

edge_img = create_edge_test_image(256)
edges_x2 = ndimage.convolve(edge_img, sobel_x)
edges_y2 = ndimage.convolve(edge_img, sobel_y)
edges_mag2 = np.sqrt(edges_x2**2 + edges_y2**2)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(edge_img, cmap='gray')
axes[0].set_title('Test Pattern')
axes[0].axis('off')

axes[1].imshow(edges_mag2, cmap='hot')
axes[1].set_title('Edge Magnitude')
axes[1].axis('off')

# Thresholded edges
threshold = np.percentile(edges_mag2, 85)
axes[2].imshow(edges_mag2 > threshold, cmap='gray')
axes[2].set_title(f'Thresholded Edges (>{threshold:.0f})')
axes[2].axis('off')

plt.suptitle('Edge Detection on Structured Patterns', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

Key concepts covered:
1. **Images are arrays**: Height x Width x Channels, with pixel values 0-255.
2. **RGB channels** can be separated and analyzed independently.
3. **Basic operations** (resize, crop, rotate, flip) are building blocks for data augmentation.
4. **Color spaces** (RGB, Grayscale, HSV, Binary) serve different purposes.
5. **Sobel edge detection** uses convolution kernels to find intensity gradients.